# Pipeline Farma Help — RAG

Este notebook ejecuta únicamente el flujo del RAG. Toda la implementación vive en ``src/farma_help/rag`` y se consume como librería.

Requisitos previos:
- Instalar dependencias: ``pip install -r requirements.txt``
- Configurar ``.env`` en la raíz del repositorio con ``GEMINI_API_KEY``, ``GEMINI_MODEL`` y ``GEMINI_EMBEDDING_MODEL``.

In [ ]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent

# Permitir importar controles (raíz) y farma_help (src).
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "src"))

from farma_help.configuracion import cargar_entorno, cargar_rutas, variable_entorno

cargar_entorno()

API_KEY = variable_entorno("GEMINI_API_KEY")
MODELO_LLM = variable_entorno("GEMINI_MODEL", "gemini-3.5-flash-lite")
MODELO_EMBEDDING = variable_entorno("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")

rutas = cargar_rutas()
print("Variables de entorno cargadas.")
print(f"Políticas: {rutas.politicas}")

## 1. Construir el servicio RAG

Esta celda lee las políticas, crea fragmentos y chunks, genera embeddings con Gemini, construye los índices FAISS por nivel de acceso y carga el reranker y el generador.

In [ ]:
from farma_help.rag.servicio import ServicioRAG

rag = ServicioRAG.construir(
    carpeta_politicas=rutas.politicas,
    api_key=API_KEY,
    modelo_llm=MODELO_LLM,
    modelo_embeddings=MODELO_EMBEDDING,
    device="cpu",
    usar_reranking=True,
)
print("✅ Servicio RAG construido.")

## 2. Recuperar chunks de prueba

Verifica que la recuperación semántica y los permisos por rol funcionan sin invocar al LLM.

In [ ]:
PREGUNTA_PRUEBA = (
    "¿Cuál es el plazo para reclamar un faltante y desde cuándo se cuenta?"
)
ROL_PRUEBA = "operador"

candidatos = rag.recuperar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=5)

print(f"Pregunta: {PREGUNTA_PRUEBA}")
print(f"Rol: {ROL_PRUEBA}\n")

for numero, c in enumerate(candidatos, start=1):
    print(f"{numero}. {c['fragmento_id']}")
    print(f"   Título: {c['titulo']}")
    print(f"   Clasificación: {c['clasificacion']}")
    print(f"   Similitud: {c.get('similitud', 0):.4f}")
    if "puntuacion_reranking" in c:
        print(f"   Reranking: {c['puntuacion_reranking']:.4f}")
    print()

## 3. Consulta completa (recuperación + reranking + generación)

Ejecuta el pipeline end-to-end y muestra la respuesta generada por Gemini junto con las fuentes utilizadas.

In [ ]:
resultado = rag.consultar(PREGUNTA_PRUEBA, rol=ROL_PRUEBA, top_k=10, top_contexto=3)

print("RESPUESTA:\n")
print(resultado["respuesta"])
print("\n--- FUENTES ---")
for numero, fuente in enumerate(resultado["fuentes"], start=1):
    print(f"{numero}. {fuente['fragmento_id']} — {fuente['titulo']}")

## 4. Consultas adicionales

Modifica la pregunta y el rol para probar otras consultas.

In [ ]:
pregunta = "¿Cuándo puede un cliente solicitar la anulación de un reclamo?"
rol = "operador"

respuesta = rag.consultar(pregunta, rol=rol, top_k=10, top_contexto=3)
print(f"Pregunta: {pregunta}")
print(f"Rol: {rol}\n")
print(respuesta["respuesta"])